<a href="https://colab.research.google.com/github/EAwoyemi110/Ai-job-market-dataset-Analysis/blob/main/Official_Experiment_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from huggingface_hub import login
login()

In [ ]:
# Use a pipeline as a high-level helper
from transformers import pipeline

pipe = pipeline("image-text-to-text", model="google/paligemma2-3b-pt-224")

In [ ]:
from transformers import AutoModel, PaliGemmaForConditionalGeneration
#Condition 1: visual only
#Loads SigLip Vision model, the "ViT-SO400M" architecture,
model_vit = AutoModel.from_pretrained("google/siglip-so400m-patch14-384")

#Condition num. 2
model_vlm = PaliGemmaForConditionalGeneration.from_pretrained("google/paligemma2-3b-pt-224")

In [ ]:
#similar to savietto et. al, we will generate 448 x 448 pixel synthetic, controlled geometric images
#to isolate independent axes for color, shape, and conjunction attributes, for VLM concept probing
from PIL import Image, ImageDraw
import random

def make_image(num_objects, canvas_size=384, seed=0):
  random.seed(seed)
  img = Image.new("RGB", (canvas_size, canvas_size), "white")
  draw = ImageDraw.Draw(img)
  objects = [] #main idea: shape, color, box
  shapes = ["circle", "square", "triangle"]
  colors = ["yellow", "green", "purple"]

  for i in range(num_objects):
    shape = random.choice(shapes)

    color = random.choice(colors)
    x, y = random.randint(20, canvas_size-60), random.randint(20, canvas_size-60)
    size = 40
    bbox = [x, y, x+size, y+size]

    if shape == "circle":
      draw.ellipse(bbox, fill=color)
    elif shape == "square":
      draw.rectangle(bbox, fill=color)
    else:
      draw.polygon([(x, y+size), (x+size/2, y), (x+size, y+size)], fill=color)
    objects.append({"shape": shape, "color": color, "bbox": bbox })

  return img, objects

    #gives exact pixel bounding box for each object


In [ ]:
def bbox_to_patch_indices(bbox, canvas_size=384, patch_size=14):
    x1, y1, x2, y2 = bbox
    grid_size = canvas_size // patch_size
    px1, py1 = x1 // patch_size, y1 // patch_size
    px2, py2 = x2 // patch_size, y2 // patch_size

    indices = []
    for py in range(py1, min(py2 + 1, grid_size)):
        for px in range(px1, min(px2 + 1, grid_size)):
            indices.append(py * grid_size + px)  # flattened patch index
    return indices



In [ ]:
import torch
from transformers import AutoImageProcessor
processor = AutoImageProcessor.from_pretrained("google/siglip-so400m-patch14-384")

inputs = processor(images=img, return_tensors="pt")

with torch.no_grad():
    _ = model_vit(**inputs)          # populates activations_vit
    _ = vision_tower(inputs["pixel_values"])  # populates activations_vlm

In [ ]:
import torch.nn.functional as F

def compute_rdm(layer_activations, objects):
    # layer_activations: [num_patches, hidden_dim]
    object_vectors = []
    for obj in objects:
        idx = obj["patch_indices"]
        vec = layer_activations[idx].mean(dim=0)  # average patches belonging to this object
        object_vectors.append(vec)
    object_vectors = torch.stack(object_vectors)  # [num_objects, hidden_dim]

    normed = F.normalize(object_vectors, dim=1)
    similarity = normed @ normed.T   # [num_objects, num_objects] cosine similarity
    rdm = 1 - similarity              # cosine distance
    return rdm, similarity

In [ ]:
results = []
for num_objects in [2, 4, 8, 16, 32]:
    for img_idx in range(N_IMAGES_PER_COUNT):
        img, objects = make_image(num_objects, seed=img_idx)
        for obj in objects:
            obj["patch_indices"] = bbox_to_patch_indices(obj["bbox"])

        run_through_both_conditions(img)  # populates activations dicts

        for layer_name, layer_acts in activations_vit.items():
            rdm, sim = compute_rdm(layer_acts, objects)
            mean_sim = sim[~torch.eye(len(objects), dtype=bool)].mean().item()
            results.append({
                "condition": "vit_only", "layer": layer_name,
                "num_objects": num_objects, "image_id": img_idx,
                "mean_similarity": mean_sim, "rdm": rdm.tolist()
            })
        # repeat for activations_vlm with condition="vlm_no_prompt"

import json
with open("results/exp1/raw_results.json", "w") as f:
    json.dump(results, f)

In [ ]:
#We will also use 32 images from a public dataset